# Proyecto final 

## Puntos a cubrir:

● Transformación y limpieza profunda de los datos.
● Análisis descriptivo de los datos.
● Análisis estadístico de los datos.
● Visualización de los datos.
● Dashboard operativo.
● Informe explicativo del análisis.

In [40]:
import pandas as pd

pd.set_option('display.max_columns', None)        # No oculta columnas
pd.set_option('display.max_colwidth', None)       # No corta el contenido de cada celda
pd.set_option('display.width', 2000)              # Evita saltos de línea por ancho
pd.set_option('display.expand_frame_repr', False) # Fuerza una sola línea por fila

In [41]:
# Carga un fichero en formato CSV, Excel o JSON y devuelve un DataFrame
def cargar_fichero(ruta_fichero: str, tipo: str) -> pd.DataFrame:
    """
    Carga un fichero y devuelve un DataFrame.
    tipo: 'csv', 'excel', 'json'
    """
    tipo = tipo.lower()

    if tipo == "csv":
        return pd.read_csv(ruta_fichero)

    elif tipo == "excel":
        return pd.read_excel(ruta_fichero, engine="openpyxl")

    elif tipo == "json":
        return pd.read_json(ruta_fichero)

    else:
        raise ValueError(f"Tipo de fichero no soportado: {tipo}")
    

# Muestra las primeras filas de un DataFrame con un título identificativo
def ver_df(nombre: str, df):
    print(f"\n===== {nombre} =====")
    print(df.head(7))


# Imprime la lista de columnas de un DataFrame con un título identificativo
def ver_columnas(nombre: str, df):
    print(f"\n===== COLUMNAS DE {nombre} (total: {len(df.columns)}) =====")
    for col in df.columns:
        print(f"- {col}")


# Elimina un DataFrame de memoria si existe
def eliminar_df(nombre: str, contexto: dict):
    """
    Elimina un DataFrame del diccionario de variables (normalmente globals() o locals()).
    """
    if nombre in contexto:
        del contexto[nombre]
        print(f"DataFrame '{nombre}' eliminado.")
    else:
        print(f"'{nombre}' no existe en el contexto.")


# Elimina columnas de un DataFrame, mostrando cuáles borra y el resultado final
def eliminar_columnas(df, columnas: list):
    """
    Elimina columnas específicas de un DataFrame e imprime las columnas eliminadas.
    columnas: lista de columnas a eliminar
    """
    print("\n===== ELIMINANDO COLUMNAS =====")
    for col in columnas:
        print(f"- {col}")

    return df.drop(columns=columnas, errors="ignore")



## Análisis previo

Este análisis nos permitirá concentrar en un vistazo la información que tenemos en los dataset y que enfoque darle para obtener un dashboard sencillo y funcional.

In [42]:

# Muestra de datos por DataFrame

df_distribution_center = cargar_fichero("data/distribution_centers.json", "json")
ver_df("df_distribution_center", df_distribution_center)

df_events = cargar_fichero("data/events.csv", "csv")
ver_df("df_events", df_events)

df_inventory_items = cargar_fichero("data/inventory_items.csv", "csv")
ver_df("df_inventory_items", df_inventory_items)

df_order_items = cargar_fichero("data/order_items.csv", "csv")
ver_df("df_order_items", df_order_items)

df_orders = cargar_fichero("data/orders.csv", "csv")
ver_df("df_orders", df_orders)

df_products = cargar_fichero("data/products.xlsx", "excel")
ver_df("df_products", df_products)

df_users = cargar_fichero("data/users.xlsx", "excel")
ver_df("df_users", df_users)


===== df_distribution_center =====
   id                                         name  latitude  longitude
0   1                                   Memphis TN   35.1174   -89.9711
1   2                                   Chicago IL   41.8369   -87.6847
2   3                                   Houston TX   29.7604   -95.3698
3   4                               Los Angeles CA   34.0500  -118.2500
4   5                               New Orleans LA   29.9500   -90.0667
5   6  Port Authority of New York/New Jersey NY/NJ   40.6340   -73.7834
6   7                              Philadelphia PA   39.9500   -75.1667

===== df_events =====
        id  user_id  sequence_number                            session_id                 created_at       ip_address          city      state postal_code browser traffic_source      uri event_type
0  2198523      NaN                3  83889ed2-2adc-4b9a-af5d-154f6998e778  2021-06-17 17:30:00+00:00    138.143.9.202     São Paulo  São Paulo   02675-031  Chrome   


La relación entre dataset:

USERS ───┐
         │
         ▼
     ORDERS ────┐
                 │
                 ▼
            ORDER_ITEMS ──────┐
                               │
                               ▼
                         INVENTORY_ITEMS ───── PRODUCTS
                               │
                               ▼
                     DISTRIBUTION_CENTER

        EVENTS ────(por user_id , aunque en la muestra se observan valores NaN lo cual genera inconsistencia en la información )──► FUNNEL

Posibles resultados:

1) Dashboard de rendimiento logístico y distribución
Objetivo: entender cómo se comporta la red logística, qué centros distribuyen más, dónde hay cuellos de botella y cómo se mueve el inventario.
2) Dashboard de comportamiento del cliente y calidad del funnel
Objetivo: entender cómo navegan los usuarios, qué eventos generan cancelaciones, qué canales traen tráfico y cómo se comportan los compradores.
3) Dashboard de performance comercial y catálogo
Objetivo: analizar productos, precios, márgenes, categorías y comportamiento de ventas.

In [43]:

# Lista de columnas por DataFrame

ver_columnas("df_distribution_center", df_distribution_center)

ver_columnas("df_events", df_events)

ver_columnas("df_inventory_items", df_inventory_items)

ver_columnas("df_order_items", df_order_items)

ver_columnas("df_orders", df_orders)

ver_columnas("df_products", df_products)

ver_columnas("df_users", df_users)


===== COLUMNAS DE df_distribution_center (total: 4) =====
- id
- name
- latitude
- longitude

===== COLUMNAS DE df_events (total: 13) =====
- id
- user_id
- sequence_number
- session_id
- created_at
- ip_address
- city
- state
- postal_code
- browser
- traffic_source
- uri
- event_type

===== COLUMNAS DE df_inventory_items (total: 12) =====
- id
- product_id
- created_at
- sold_at
- cost
- product_category
- product_name
- product_brand
- product_retail_price
- product_department
- product_sku
- product_distribution_center_id

===== COLUMNAS DE df_order_items (total: 11) =====
- id
- order_id
- user_id
- product_id
- inventory_item_id
- status
- created_at
- shipped_at
- delivered_at
- returned_at
- sale_price

===== COLUMNAS DE df_orders (total: 9) =====
- order_id
- user_id
- status
- gender
- created_at
- returned_at
- shipped_at
- delivered_at
- num_of_item

===== COLUMNAS DE df_products (total: 9) =====
- id
- cost
- category
- name
- brand
- retail_price
- department
- sku
- dis

### Dataset seleccionados

Me quedo con los dataset: orders, order_items y products para generar un **Dashboard de performance comercial**

===== COLUMNAS DE df_orders (total: 8) =====
- order_id
- status
- gender
- created_at
- returned_at
- shipped_at
- delivered_at
- num_of_item

===== COLUMNAS DE df_order_items (total: 9) =====
- id
- order_id
- product_id
- status
- created_at
- shipped_at
- delivered_at
- returned_at
- sale_price

===== COLUMNAS DE df_products (total: 7) =====
- id
- cost
- category
- name
- brand
- retail_price
- department


# Transformación y limpieza profunda de los datos.

In [44]:
# Elimina los DataFrame que no se usan

eliminar_df("df_distribution_center", globals())

eliminar_df("df_events", globals())

eliminar_df("df_inventory_items", globals())

eliminar_df("df_users", globals())

DataFrame 'df_distribution_center' eliminado.
DataFrame 'df_events' eliminado.
DataFrame 'df_inventory_items' eliminado.
DataFrame 'df_users' eliminado.


In [45]:
# Elimina columnas prescindible por DataFrame (no aportan nada al dashboard)

df_orders = eliminar_columnas(df_orders,["user_id"] )

df_order_items = eliminar_columnas(df_order_items,["user_id","inventory_item_id"] )

df_products = eliminar_columnas(df_products,["sku","distribution_center_id"] )

# Ver datos
ver_df("df_orders", df_orders)

ver_df("df_order_items", df_order_items)

ver_df("df_products", df_products)



===== ELIMINANDO COLUMNAS =====
- user_id

===== ELIMINANDO COLUMNAS =====
- user_id
- inventory_item_id

===== ELIMINANDO COLUMNAS =====
- sku
- distribution_center_id

===== df_orders =====
   order_id     status gender                 created_at returned_at shipped_at delivered_at  num_of_item
0         8  Cancelled      F  2022-10-20 10:03:00+00:00         NaN        NaN          NaN            3
1        60  Cancelled      F  2023-01-20 02:12:00+00:00         NaN        NaN          NaN            1
2        64  Cancelled      F  2021-12-06 09:11:00+00:00         NaN        NaN          NaN            1
3        89  Cancelled      F  2020-08-13 09:58:00+00:00         NaN        NaN          NaN            1
4       102  Cancelled      F  2023-01-17 08:17:00+00:00         NaN        NaN          NaN            2
5       117  Cancelled      F  2023-07-31 13:25:00+00:00         NaN        NaN          NaN            1
6       143  Cancelled      F  2020-04-21 02:59:00+00:00         